In [29]:
import pandas as pd
import time
import io
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# =================================================================
# CONFIGURACIÓN: CAMBIA ESTO PARA CADA LIGA
# =================================================================
URL_LIGA = "https://www.whoscored.com/regions/74/tournaments/22/seasons/10329/stages/23414/playerstatistics/france-ligue-1-2024-2025"
NOMBRE_CSV_RAW = "ligue1_stats_completa_24_25.csv"  # Nombre del archivo sucio
CARPETA = r"c:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Rating"
# =================================================================

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def extraer_todas_las_paginas(url):
    driver.get(url)
    try:
        cookie_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Agree')] | //button[contains(text(), 'Aceptar')]"))
        )
        cookie_btn.click()
        time.sleep(2)
    except: pass

    try:
        all_players = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.LINK_TEXT, "All players")))
        driver.execute_script("arguments[0].click();", all_players)
        time.sleep(3)
    except: pass

    lista_dataframes = []
    page_count = 1
    
    while True:
        try:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.ID, "top-player-stats-summary-grid")))
            html_tabla = driver.find_element(By.ID, "top-player-stats-summary-grid").get_attribute('outerHTML')
            df_temporal = pd.read_html(io.StringIO(html_tabla))[0]
            lista_dataframes.append(df_temporal)
            
            next_button = driver.find_element(By.ID, "next")
            if "disabled" in next_button.get_attribute("class"): break
            
            print(f"Página {page_count} extraída...")
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(3)
            page_count += 1
        except: break

    return pd.concat(lista_dataframes, ignore_index=True) if lista_dataframes else None

try:
    df_resultado = extraer_todas_las_paginas(URL_LIGA)
    if df_resultado is not None:
        import os
        ruta_completa = os.path.join(CARPETA, NOMBRE_CSV_RAW)
        df_resultado.drop_duplicates().to_csv(ruta_completa, index=False, encoding='utf-8-sig')
        print(f"¡Listo! Archivo crudo guardado en: {ruta_completa}")
finally:
    driver.quit()

Página 1 extraída...
Página 2 extraída...
Página 3 extraída...
Página 4 extraída...
Página 5 extraída...
Página 6 extraída...
Página 7 extraída...
Página 8 extraída...
Página 9 extraída...
Página 10 extraída...
Página 11 extraída...
Página 12 extraída...
Página 13 extraída...
Página 14 extraída...
Página 15 extraída...
Página 16 extraída...
Página 17 extraída...
Página 18 extraída...
Página 19 extraída...
Página 20 extraída...
Página 21 extraída...
Página 22 extraída...
Página 23 extraída...
Página 24 extraída...
Página 25 extraída...
Página 26 extraída...
Página 27 extraída...
Página 28 extraída...
Página 29 extraída...
Página 30 extraída...
Página 31 extraída...
Página 32 extraída...
Página 33 extraída...
Página 34 extraída...
Página 35 extraída...
Página 36 extraída...
Página 37 extraída...
Página 38 extraída...
Página 39 extraída...
Página 40 extraída...
Página 41 extraída...
Página 42 extraída...
Página 43 extraída...
Página 44 extraída...
Página 45 extraída...
Página 46 extraída.

In [30]:
import pandas as pd
import re
import os

# =================================================================
# CONFIGURACIÓN: CAMBIA ESTO PARA LIMPIAR EL ARCHIVO
# =================================================================
mi_carpeta = r"c:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Rating"
archivo_entrada = "ligue1_stats_completa_24_25.csv" # Debe coincidir con el del Código 1
archivo_salida = "LIGUE1_24_25_LIMPIO_FINAL.csv"  # Nombre del archivo ya profesional
# =================================================================

ruta_input = os.path.join(mi_carpeta, archivo_entrada)
ruta_output = os.path.join(mi_carpeta, archivo_salida)

try:
    df = pd.read_csv(ruta_input)
    
    def limpiar_jugador_equipo(texto):
        texto = str(texto)
        # 1. Quitar ranking del inicio
        texto_sin_numero = re.sub(r'^\d+', '', texto).strip()
        # 2. Cortar en la primera coma
        parte_principal = texto_sin_numero.split(',')[0].strip()
        # 3. Separar Equipo por Mayúscula final
        match_equipo = re.search(r'([A-Z][a-z]+(?:\s[A-Z][a-z]+)*)$', parte_principal)
        
        if match_equipo:
            equipo = match_equipo.group(1).strip()
            nombre = parte_principal.replace(equipo, "").strip()
            return nombre, equipo
        return parte_principal, "Desconocido"

    # Aplicar lógica
    df[['Player_Name', 'Team']] = df['Player'].apply(lambda x: pd.Series(limpiar_jugador_equipo(x)))
    
    # Formateo de tipos de datos
    for col in ['Rating', 'Mins', 'Goals', 'Assists']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Selección final de columnas (Aquí quitas lo que no quieras)
    columnas_finales = ['Player_Name', 'Team', 'Rating', 'Mins', 'Goals', 'Assists', 'Apps']
    df_final = df[columnas_finales]

    # Guardar
    df_final.to_csv(ruta_output, index=False, encoding='utf-8-sig')
    
    print(f"--- LIMPIEZA EXITOSA: {archivo_salida} ---")
    print(df_final.head(5))

except Exception as e:
    print(f"Error: {e}")

--- LIMPIEZA EXITOSA: LIGUE1_24_25_LIMPIO_FINAL.csv ---
           Player_Name         Team  Rating  Mins  Goals  Assists   Apps
0  Benjamin Bourigeaud       Rennes    7.99    81    1.0      NaN      1
1       Arnau TenasPSG  Desconocido    7.81    90    NaN      1.0      1
2          Logan Costa     Toulouse    7.73    90    NaN      NaN      1
3   Ousmane DembéléPSG  Desconocido    7.65  1737   21.0      6.0  20(9)
4        Ismail Jakobs       Monaco    7.55   180    NaN      NaN      2
